# 09 — Análisis NLP: BERTopic + Clasificación Zero-Shot

Ejecuta el pipeline completo de NLP sobre `scam_us_FINAL_v8.csv`.

**Fase 1 — BERTopic:** tópicos emergentes no supervisados (~5-10 min).
**Fase 2 — BART-MNLI zero-shot:** clasificación en 14 categorías (~30-90 min).

## Antes de Run All

- Mac enchufado.
- `caffeinate -dimsu` corriendo en otra terminal.
- Kernel del notebook: **Python (tfm-nlp)**.


## Verificación de que el entorno está bien

In [1]:
# Verificación rápida de que todo está OK antes de empezar
import sys
print(f"Python: {sys.version}")

import numpy, torch, transformers, sentence_transformers, bertopic, pandas
print(f"numpy:                 {numpy.__version__}")
print(f"torch:                 {torch.__version__}")
print(f"transformers:          {transformers.__version__}")
print(f"sentence-transformers: {sentence_transformers.__version__}")
print(f"bertopic:              {bertopic.__version__}")
print(f"pandas:                {pandas.__version__}")
print()
print("✓ Entorno listo")


Python: 3.11.9 | packaged by conda-forge | (main, Apr 19 2024, 18:45:13) [Clang 16.0.6 ]
numpy:                 1.26.4
torch:                 2.2.0
transformers:          4.41.2
sentence-transformers: 3.4.1
bertopic:              0.17.4
pandas:                3.0.3

✓ Entorno listo


## Carga del corpus

In [2]:
import pandas as pd
import numpy as np
import os
import re
import time

pd.set_option('display.max_colwidth', None)

df = pd.read_csv("../data/raw/scam_us_FINAL_v8.csv")
print(f"Tweets cargados: {len(df)}")

# Preparar texto limpio
URL_RE = re.compile(r"https?://\S+")
MENTION_RE = re.compile(r"@\w+")

def clean_for_analysis(text):
    if not isinstance(text, str):
        return ""
    t = URL_RE.sub("", text)
    t = MENTION_RE.sub("", t)
    return re.sub(r"\s+", " ", t).strip()

if "clean_text" not in df.columns or df["clean_text"].isna().all():
    df["clean_text"] = df["text"].apply(clean_for_analysis)
else:
    df["clean_text"] = df["clean_text"].fillna("").apply(lambda x: x if x else "")
    mask_empty = df["clean_text"] == ""
    df.loc[mask_empty, "clean_text"] = df.loc[mask_empty, "text"].apply(clean_for_analysis)

# Filtrar tweets muy cortos
df = df[df["clean_text"].str.len() >= 20].reset_index(drop=True)
print(f"Tras filtrar textos cortos: {len(df)}")

docs = df["clean_text"].tolist()


Tweets cargados: 1928
Tras filtrar textos cortos: 1928


## FASE 1 — BERTopic

⏳ ~5-10 minutos. Descarga inicial del modelo all-MiniLM-L6-v2 (~80 MB).


In [3]:
print("⏳ Iniciando BERTopic...")
t0 = time.time()

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# Para BERTopic 0.16+ con sentence-transformers 2.7, mejor pasar el nombre del modelo
# y dejar que BERTopic gestione los submodelos por defecto.
topic_model = BERTopic(
    embedding_model="all-MiniLM-L6-v2",
    min_topic_size=30,           # adaptado a corpus de ~2k
    nr_topics="auto",            # reduce automáticamente tópicos similares
    calculate_probabilities=False,
    language="english",
    verbose=True,
)

print("⏳ Calculando embeddings y agrupando...")
topics, _ = topic_model.fit_transform(docs)

t_bertopic = time.time() - t0
n_topics = len(set(topics)) - (1 if -1 in topics else 0)
n_outliers = sum(1 for t in topics if t == -1)
print(f"\n✓ BERTopic completado en {t_bertopic/60:.1f} min")
print(f"  Tópicos encontrados: {n_topics}")
print(f"  Tweets en outliers (-1): {n_outliers}")


2026-05-19 03:34:35,596 - BERTopic - Embedding - Transforming documents to embeddings.


⏳ Iniciando BERTopic...
⏳ Calculando embeddings y agrupando...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/61 [00:00<?, ?it/s]

2026-05-19 03:36:07,978 - BERTopic - Embedding - Completed ✓
2026-05-19 03:36:07,981 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-19 03:36:36,726 - BERTopic - Dimensionality - Completed ✓
2026-05-19 03:36:36,729 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-19 03:36:36,938 - BERTopic - Cluster - Completed ✓
2026-05-19 03:36:36,939 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-05-19 03:36:37,362 - BERTopic - Representation - Completed ✓
2026-05-19 03:36:37,363 - BERTopic - Topic reduction - Reducing number of topics
2026-05-19 03:36:37,390 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-19 03:36:37,569 - BERTopic - Representation - Completed ✓
2026-05-19 03:36:37,572 - BERTopic - Topic reduction - Reduced number of topics from 12 to 12



✓ BERTopic completado en 2.0 min
  Tópicos encontrados: 11
  Tweets en outliers (-1): 337


In [4]:
# Información detallada de los tópicos
topic_info = topic_model.get_topic_info()
print("=== INFO DE TÓPICOS ===")
print(topic_info.head(25))


=== INFO DE TÓPICOS ===
    Topic  Count                          Name  \
0      -1    337            -1_to_the_and_scam   
1       0    388      0_crypto_scam_to_romance   
2       1    309         1_ponzi_scheme_the_is   
3       2    190       2_phishing_email_and_it   
4       3    142          3_text_scam_toll_the   
5       4    136           4_rug_pull_the_this   
6       5    122       5_paypal_scam_to_emails   
7       6     97             6_job_scam_to_for   
8       7     89   7_delivery_holiday_fake_and   
9       8     41      8_text_fake_messages_and   
10      9     39  9_google_behind_lawsuit_text   
11     10     38       10_app_cash_scam_people   

                                                                                 Representation  \
0                                   [to, the, and, scam, in, fake, smishing, of, you, phishing]   
1                                    [crypto, scam, to, romance, money, and, in, for, you, the]   
2                           

In [5]:
# Asignar tópico y keywords a cada tweet
df["bertopic_id"] = topics

# Diccionario topic_id → keywords
topic_keywords = {}
for tid in set(topics):
    if tid == -1:
        topic_keywords[tid] = "outliers"
        continue
    words = topic_model.get_topic(tid)
    if words:
        topic_keywords[tid] = ", ".join([w for w, _ in words[:8]])
    else:
        topic_keywords[tid] = ""

df["bertopic_keywords"] = df["bertopic_id"].map(topic_keywords)

print("=== RESUMEN BERTOPIC ===\n")
for tid in sorted(set(topics)):
    n = (df["bertopic_id"] == tid).sum()
    kw = topic_keywords.get(tid, "")
    print(f"  Tópico {tid:>3} ({n:>4} tweets):  {kw[:80]}")


=== RESUMEN BERTOPIC ===

  Tópico  -1 ( 337 tweets):  outliers
  Tópico   0 ( 388 tweets):  crypto, scam, to, romance, money, and, in, for
  Tópico   1 ( 309 tweets):  ponzi, scheme, the, is, to, and, of, it
  Tópico   2 ( 190 tweets):  phishing, email, and, it, to, your, the, you
  Tópico   3 ( 142 tweets):  text, scam, toll, the, to, messages, and, from
  Tópico   4 ( 136 tweets):  rug, pull, the, this, is, be, to, will
  Tópico   5 ( 122 tweets):  paypal, scam, to, emails, the, draining, sneaky, that
  Tópico   6 (  97 tweets):  job, scam, to, for, the, linkedin, are, of
  Tópico   7 (  89 tweets):  delivery, holiday, fake, and, scams, package, your, are
  Tópico   8 (  41 tweets):  text, fake, messages, and, candace, her, the, she
  Tópico   9 (  39 tweets):  google, behind, lawsuit, text, china, lighthouse, smishing, operation
  Tópico  10 (  38 tweets):  app, cash, scam, people, niggas, that, ass, block


In [6]:
# Ejemplos por tópico
print("=== EJEMPLOS POR TÓPICO ===\n")
shown = 0
for tid in sorted(set(topics)):
    if shown >= 12:
        break
    subset = df[df["bertopic_id"] == tid]
    kw = topic_keywords.get(tid, "")
    print(f"--- TÓPICO {tid} ({len(subset)} tweets) ---")
    print(f"    Keywords: {kw}")
    for _, row in subset.head(2).iterrows():
        print(f"    • {str(row['text'])[:200]}")
    print()
    shown += 1


=== EJEMPLOS POR TÓPICO ===

--- TÓPICO -1 (337 tweets) ---
    Keywords: outliers
    • 1️⃣  THREAD: The Truth About Crypto Wallets &amp; Email Addresses – Why Scammers Know Yours (Even If Your Wallet Doesn't)
1/15
Crypto fam, there's a common myth/misunderstanding: "If I get a scam emai
    • I bet 75% of premium accounts are bots, and the money to pay creator revenue is coming from other bots paying their own premium subscriptions. 

X creator payouts is a Ponzi scheme built by bots payin

--- TÓPICO 0 (388 tweets) ---
    Keywords: crypto, scam, to, romance, money, and, in, for
    • ⚠️ SCAM WARNING: #Xdcbit runs a Ponzi scheme without a license.
🚫 Stop all transactions and warn others.
📩 Contact trusted #CryptoRecovery experts for safe help.
#CryptoScam #FundsRecovery #RecoverCry
    • Pick the scandal Diego Pavia will be a key piece of in 8 years:

- Real estate scam
- Bank robbery
- Crypto scam
- Human Trafficking 
- Tax Fraud

--- TÓPICO 1 (309 tweets) ---
    Keywords: ponzi, 

## Guardado intermedio (post-BERTopic)

Por si la fase de zero-shot falla, al menos BERTopic queda guardado.


In [7]:
df.to_csv("../data/raw/scam_us_v8_with_bertopic.csv", index=False, encoding="utf-8")
print(f"✓ Guardado intermedio: scam_us_v8_with_bertopic.csv ({len(df)} filas)")


✓ Guardado intermedio: scam_us_v8_with_bertopic.csv (1928 filas)


## FASE 2 — Clasificación Zero-Shot con BART-MNLI

⏳ Esta es la parte lenta (30-90 min en CPU). Las 14 categorías de tu memoria del TFM + "no relacionado":


In [8]:
CANDIDATE_LABELS = [
    "investment scam or cryptocurrency fraud",
    "romance scam",
    "phishing or identity theft",
    "government impersonation scam (IRS, USPS)",
    "bank fraud or wire transfer fraud",
    "payment app scam (Zelle, Venmo, Cash App)",
    "Ponzi or pyramid scheme",
    "tech support scam",
    "employment or job scam",
    "charity or donation scam",
    "insurance fraud",
    "corporate or securities fraud",
    "tax fraud or tax evasion",
    "not related to financial fraud",
]

LABEL_TO_CODE = {
    "investment scam or cryptocurrency fraud": "investment_crypto",
    "romance scam": "romance",
    "phishing or identity theft": "phishing_identity",
    "government impersonation scam (IRS, USPS)": "gov_impersonation",
    "bank fraud or wire transfer fraud": "bank_wire",
    "payment app scam (Zelle, Venmo, Cash App)": "payment_app",
    "Ponzi or pyramid scheme": "ponzi_pyramid",
    "tech support scam": "tech_support",
    "employment or job scam": "employment",
    "charity or donation scam": "charity",
    "insurance fraud": "insurance",
    "corporate or securities fraud": "corporate",
    "tax fraud or tax evasion": "tax",
    "not related to financial fraud": "not_related",
}

print(f"Categorías candidatas: {len(CANDIDATE_LABELS)}")
for i, lbl in enumerate(CANDIDATE_LABELS, 1):
    print(f"  {i:2d}. {lbl}")


Categorías candidatas: 14
   1. investment scam or cryptocurrency fraud
   2. romance scam
   3. phishing or identity theft
   4. government impersonation scam (IRS, USPS)
   5. bank fraud or wire transfer fraud
   6. payment app scam (Zelle, Venmo, Cash App)
   7. Ponzi or pyramid scheme
   8. tech support scam
   9. employment or job scam
  10. charity or donation scam
  11. insurance fraud
  12. corporate or securities fraud
  13. tax fraud or tax evasion
  14. not related to financial fraud


In [9]:
print("⏳ Cargando modelo BART-MNLI (~1.6 GB, primera vez tarda 5-10 min)...")
t0 = time.time()

from transformers import pipeline
import torch

# En MacBook Air Intel → CPU
device = -1
print(f"   Dispositivo: CPU")

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device,
)

t_load = time.time() - t0
print(f"✓ Modelo cargado en {t_load/60:.1f} min")


⏳ Cargando modelo BART-MNLI (~1.6 GB, primera vez tarda 5-10 min)...
   Dispositivo: CPU


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✓ Modelo cargado en 2.3 min


In [10]:
# Clasificación con guardado incremental cada 100 tweets
texts_to_classify = df["clean_text"].tolist()
n = len(texts_to_classify)
BATCH_SIZE = 8
CHECKPOINT_EVERY = 100

predictions = []
scores = []

print(f"⏳ Clasificando {n} tweets en {len(CANDIDATE_LABELS)} categorías...")
print(f"   Lote: {BATCH_SIZE} | Checkpoint cada {CHECKPOINT_EVERY}")
print(f"   ESTIMACIÓN: 30-90 min en CPU. Sé paciente.\n")

t0 = time.time()
last_print = t0

for i in range(0, n, BATCH_SIZE):
    batch = texts_to_classify[i:i+BATCH_SIZE]

    try:
        results = classifier(
            batch,
            candidate_labels=CANDIDATE_LABELS,
            multi_label=False,
        )
        if isinstance(results, dict):
            results = [results]
    except Exception as e:
        print(f"  ⚠️ Error en lote {i}: {e}")
        for _ in batch:
            predictions.append("not related to financial fraud")
            scores.append(0.0)
        continue

    for r in results:
        predictions.append(r["labels"][0])
        scores.append(r["scores"][0])

    if (time.time() - last_print) > 30 or (i + BATCH_SIZE) % CHECKPOINT_EVERY == 0:
        elapsed = time.time() - t0
        done = min(i + BATCH_SIZE, n)
        eta = (elapsed / done) * (n - done) if done > 0 else 0
        print(f"  {done}/{n} ({done/n*100:.1f}%) — {elapsed/60:.1f} min — ETA: {eta/60:.1f} min")
        last_print = time.time()

    # Checkpoint cada 100
    if (i + BATCH_SIZE) % CHECKPOINT_EVERY == 0 and (i + BATCH_SIZE) < n:
        df_partial = df.iloc[:len(predictions)].copy()
        df_partial["predicted_label"] = predictions
        df_partial["confidence_score"] = scores
        df_partial.to_csv("../data/raw/scam_us_v8_classification_checkpoint.csv", index=False)

t_classify = time.time() - t0
print(f"\n✓ Clasificación completada en {t_classify/60:.1f} min")


⏳ Clasificando 1928 tweets en 14 categorías...
   Lote: 8 | Checkpoint cada 100
   ESTIMACIÓN: 30-90 min en CPU. Sé paciente.

  8/1928 (0.4%) — 1.9 min — ETA: 449.2 min
  16/1928 (0.8%) — 3.2 min — ETA: 386.5 min
  24/1928 (1.2%) — 4.9 min — ETA: 392.4 min
  32/1928 (1.7%) — 6.4 min — ETA: 379.5 min
  40/1928 (2.1%) — 8.0 min — ETA: 376.0 min
  48/1928 (2.5%) — 9.7 min — ETA: 380.0 min
  56/1928 (2.9%) — 11.2 min — ETA: 374.3 min
  64/1928 (3.3%) — 13.2 min — ETA: 384.1 min
  72/1928 (3.7%) — 15.3 min — ETA: 393.5 min
  80/1928 (4.1%) — 17.2 min — ETA: 396.2 min
  88/1928 (4.6%) — 18.7 min — ETA: 391.6 min
  96/1928 (5.0%) — 20.2 min — ETA: 385.4 min
  104/1928 (5.4%) — 21.8 min — ETA: 382.6 min
  112/1928 (5.8%) — 23.3 min — ETA: 377.3 min
  120/1928 (6.2%) — 25.2 min — ETA: 379.5 min
  128/1928 (6.6%) — 26.7 min — ETA: 375.1 min
  136/1928 (7.1%) — 28.0 min — ETA: 369.1 min
  144/1928 (7.5%) — 29.6 min — ETA: 366.5 min
  152/1928 (7.9%) — 30.9 min — ETA: 361.3 min
  160/1928 (8.3%) 

In [11]:
# Añadir predicciones al DataFrame
df["predicted_label"] = predictions[:len(df)]
df["confidence_score"] = scores[:len(df)]
df["predicted_category"] = df["predicted_label"].map(LABEL_TO_CODE)
df["is_relevant"] = df["predicted_category"] != "not_related"

# Distribución
print("=== DISTRIBUCIÓN DE CATEGORÍAS PREDICHAS ===\n")
counts = df["predicted_category"].value_counts()
total = len(df)
for cat, n_cat in counts.items():
    pct = n_cat / total * 100
    print(f"  {cat:<22} {n_cat:>5}  ({pct:>5.1f}%)")

print(f"\nRelevantes (no 'not_related'): {df['is_relevant'].sum()} / {total} ({df['is_relevant'].mean()*100:.1f}%)")


=== DISTRIBUCIÓN DE CATEGORÍAS PREDICHAS ===

  phishing_identity        636  ( 33.0%)
  ponzi_pyramid            298  ( 15.5%)
  investment_crypto        293  ( 15.2%)
  employment               248  ( 12.9%)
  romance                  194  ( 10.1%)
  bank_wire                 57  (  3.0%)
  tech_support              43  (  2.2%)
  payment_app               37  (  1.9%)
  not_related               36  (  1.9%)
  charity                   34  (  1.8%)
  tax                       22  (  1.1%)
  corporate                 20  (  1.0%)
  insurance                  5  (  0.3%)
  gov_impersonation          5  (  0.3%)

Relevantes (no 'not_related'): 1892 / 1928 (98.1%)


In [12]:
# Distribución de confianza
print("=== CONFIANZA EN PREDICCIONES ===\n")
print(f"  Media:    {df['confidence_score'].mean():.3f}")
print(f"  Mediana:  {df['confidence_score'].median():.3f}")
print(f"  Min:      {df['confidence_score'].min():.3f}")
print(f"  Max:      {df['confidence_score'].max():.3f}")
print(f"\n  Alta confianza (≥0.7):  {(df['confidence_score'] >= 0.7).sum()}")
print(f"  Media (0.4-0.7):        {((df['confidence_score'] >= 0.4) & (df['confidence_score'] < 0.7)).sum()}")
print(f"  Baja (<0.4):            {(df['confidence_score'] < 0.4).sum()}")


=== CONFIANZA EN PREDICCIONES ===

  Media:    0.596
  Mediana:  0.619
  Min:      0.097
  Max:      0.992

  Alta confianza (≥0.7):  738
  Media (0.4-0.7):        707
  Baja (<0.4):            483


## Guardado final

In [13]:
OUTPUT = "../data/raw/scam_us_FINAL_classified.csv"
df.to_csv(OUTPUT, index=False, encoding="utf-8")

print(f"✓ Guardado: {OUTPUT}")
print(f"  Total filas: {len(df)}")
print()
print("📦 Archivos generados:")
print(f"   scam_us_v8_with_bertopic.csv         ({len(df)} filas - solo BERTopic)")
print(f"   scam_us_v8_classification_checkpoint.csv (checkpoints intermedios)")
print(f"   scam_us_FINAL_classified.csv         ({len(df)} filas - clasificado completo)")
print()
print("🚨 HAZ COPIA DE SEGURIDAD del scam_us_FINAL_classified.csv en Drive.")
print()
print("📌 LISTO PARA LA FASE DE ANÁLISIS DE TU TFM.")


✓ Guardado: ../data/raw/scam_us_FINAL_classified.csv
  Total filas: 1928

📦 Archivos generados:
   scam_us_v8_with_bertopic.csv         (1928 filas - solo BERTopic)
   scam_us_v8_classification_checkpoint.csv (checkpoints intermedios)
   scam_us_FINAL_classified.csv         (1928 filas - clasificado completo)

🚨 HAZ COPIA DE SEGURIDAD del scam_us_FINAL_classified.csv en Drive.

📌 LISTO PARA LA FASE DE ANÁLISIS DE TU TFM.


## Inspección de resultados

In [14]:
print("=== 3 EJEMPLOS DE CADA CATEGORÍA (alta confianza) ===\n")
for cat_code in df["predicted_category"].value_counts().index[:10]:
    subset = df[
        (df["predicted_category"] == cat_code) &
        (df["confidence_score"] >= 0.5)
    ].head(3)
    n_total = (df["predicted_category"] == cat_code).sum()
    print(f"--- {cat_code} ({n_total} total) ---")
    for _, row in subset.iterrows():
        print(f"  [{row['confidence_score']:.2f}] {str(row['text'])[:200]}")
    print()


=== 3 EJEMPLOS DE CADA CATEGORÍA (alta confianza) ===

--- phishing_identity (636 total) ---
  [0.56] Whether it’s vishing, quishing, smishing, or old-school spear phishing, it takes more than instinct to spot cyberattacks. https://t.co/6TRJF72NlU https://t.co/AgMGDfSyR1
  [0.59] Metro advises ExpressLanes customers to not respond to phishing text messages.

https://t.co/F4Rg6AS6eh
  [0.87] Smishing is phishing via text—and it’s getting smarter. 📱⚠️
Scammers use real names, fake “secure” links, and urgent timing.
Never click links in texts.
Check accounts only through official apps or we

--- ponzi_pyramid (298 total) ---
  [0.50] ⚠️ SCAM WARNING: #Xdcbit runs a Ponzi scheme without a license.
🚫 Stop all transactions and warn others.
📩 Contact trusted #CryptoRecovery experts for safe help.
#CryptoScam #FundsRecovery #RecoverCry
  [0.54] SDWC Financial LLC and one of its senior analysts are free of a federal lawsuit alleging they solicited investments in a $300 million real estate Ponzi

In [15]:
# Tweets clasificados como "not_related" - verifica que están bien descartados
nr_count = (df["predicted_category"] == "not_related").sum()
if nr_count > 0:
    print(f"=== EJEMPLOS DE 'not_related' ({nr_count} en total) ===\n")
    not_related = df[df["predicted_category"] == "not_related"].sample(
        min(10, nr_count), random_state=42,
    )
    for _, row in not_related.iterrows():
        print(f"  [{row['confidence_score']:.2f}] @{row['username']}")
        print(f"    {str(row['text'])[:200]}")
        print()
else:
    print("No hay tweets en 'not_related'.")


=== EJEMPLOS DE 'not_related' (36 en total) ===

  [0.97] @Kummernuss
    No it is not a Ponzi Scheme. How is the ROI? https://t.co/SybT9N2ahV

  [0.17] @megofishing13
    Got a text from one of my credit card companies. They stopped 2 separate charges as possible fraud (they were).

I don't ever use the card. As a matter of fact, it sits in my safe.
It's not attached t

  [0.55] @PayPriestess
    Nope. I dropped my car off at M&amp;R mechanic in Lancaster and picked it up when they called me. I didn't feel the need to be harassed with constant updates. I didn't have crippling anxiety about my 

  [0.97] @adastra23
    It's not some pig butchering scam, the dude really works at Smithfield.

  [0.15] @8x8
    Fraud isn’t slowing down, it’s getting smarter.

In the latest PodChats for FutureCISO, 8x8 CPaaS Product Director Igor Mostovoy explains how real-time visibility and clearer threat signals help organ

  [0.99] @ToneClooney
    that’s not an Apple Pay scam that’s just sharing conta